# Stage 3: Warehouse and Transformation (Snowflake + dbt)
1. Load the Stage 2 Parquet files into a Snowflake **raw** table.
2. Run **dbt**: a staging model (type casting, light cleanup) and two marts:
   - `mart_rating_stats`: rating statistics per category and year
   - `mart_model_data`: balanced sample per category, with labels and train/test flags for the in-category and temporal experiments
3. Run the dbt **tests** (not null, unique review IDs, ratings 1-5, valid categories).
4. Export `mart_model_data` to `data/processed/model_data.parquet`, which Stage 4 uses automatically.

**Raw data -> RAW.REVIEWS -> dbt staging view -> dbt mart tables**

## One-time setup (about 10 minutes)

**a) Snowflake account.** Create a free trial at snowflake.com. Pick any cloud/region.

**b) Key pair for login.** Snowflake no longer accepts a plain username + password for scripts, so we log in with a key pair (no MFA prompts). In a terminal (Git Bash on Windows) run:
```
openssl genrsa 2048 | openssl pkcs8 -topk8 -inform PEM -out rsa_key.p8 -nocrypt
openssl rsa -in rsa_key.p8 -pubout -out rsa_key.pub
```
Keep `rsa_key.p8` private. **Never commit it to GitHub** (add `*.p8` to `.gitignore`).

**c) Register the public key.** Open `rsa_key.pub`, copy everything between the `BEGIN` and `END` lines as one line, and run this in a Snowsight worksheet (replace the user name and key):
```
ALTER USER YOUR_USER SET RSA_PUBLIC_KEY='MIIBIjANBgkq...';
```

**d) Account identifier.** In Snowsight, click your name (bottom left) -> your account -> **Copy account identifier**. It looks like `MYORG-MYACCOUNT`.

**e) Put the dbt project in your repo** as a folder named `dbt/` (next to this notebook).

The next cells ask for the account identifier, your user name, and the path to `rsa_key.p8`.

In [ ]:
%pip install -q snowflake-connector-python pyarrow pandas dbt-snowflake

## 1. Settings

In [ ]:
import os, subprocess
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import snowflake.connector

LOAD = "sample"    # "sample" = the 10% sample (~1.8M rows, quick) | "full" = all ~18M rows (several GB to upload)
PARQUET_DIR = Path("data/processed") / ("reviews_sample.parquet" if LOAD == "sample" else "reviews.parquet")
CATEGORIES = ["Books", "Music", "Video_DVD", "Digital_Ebook"]
DB, WH = "AMAZON_REVIEWS", "REVIEWS_WH"

def need(name):
    if not os.environ.get(name):
        os.environ[name] = input(f"{name}: ").strip()

need("SNOWFLAKE_ACCOUNT")           # e.g. MYORG-MYACCOUNT
need("SNOWFLAKE_USER")
need("SNOWFLAKE_PRIVATE_KEY_PATH")  # e.g. C:/Users/me/rsa_key.p8 or /home/me/rsa_key.p8

## 2. Connect and create the database, schemas, stage, and raw table

In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    authenticator="SNOWFLAKE_JWT",
    private_key_file=os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"],
    role="ACCOUNTADMIN",
)
cur = conn.cursor()

def run(sql):
    cur.execute(sql)
    return cur

print(run("select current_user(), current_account(), current_role()").fetchall())

In [ ]:
statements = [
    f"CREATE WAREHOUSE IF NOT EXISTS {WH} WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE",
    f"CREATE DATABASE IF NOT EXISTS {DB}",
    f"USE WAREHOUSE {WH}",
    f"USE DATABASE {DB}",
    "CREATE SCHEMA IF NOT EXISTS RAW",
    "CREATE SCHEMA IF NOT EXISTS ANALYTICS",
    """CREATE OR REPLACE TABLE RAW.REVIEWS (
        review_id          STRING,
        product_id         STRING,
        customer_id        NUMBER(38,0),
        category           STRING,
        star_rating        NUMBER(2,0),
        helpful_votes      NUMBER(10,0),
        total_votes        NUMBER(10,0),
        verified_purchase  BOOLEAN,
        review_date        DATE,
        review_year        NUMBER(4,0),
        review_text        STRING,
        word_count         NUMBER(10,0)
    )""",
    "CREATE OR REPLACE FILE FORMAT RAW.PARQUET_FMT TYPE = PARQUET",
    "CREATE OR REPLACE STAGE RAW.REVIEWS_STAGE FILE_FORMAT = RAW.PARQUET_FMT",
]
for sql in statements:
    run(sql)
print("Created warehouse, database, schemas RAW and ANALYTICS, table RAW.REVIEWS, and stage.")

## 3. Upload the Parquet files and load them
Spark stored the category in the folder name (`category=Books/`), not inside the files, so we upload each category to its own stage folder and fill in the `category` column after each load.

In [ ]:
for cat in CATEGORIES:
    folder = (PARQUET_DIR / f"category={cat}").resolve()
    files = list(folder.glob("*.parquet"))
    assert files, f"No Parquet files found in {folder}"
    print(f"Uploading {cat}: {len(files)} files ...")
    run(f"PUT 'file://{folder.as_posix()}/*.parquet' @RAW.REVIEWS_STAGE/{cat}/ AUTO_COMPRESS=FALSE PARALLEL=8 OVERWRITE=TRUE")
print("Upload done.")

In [ ]:
for cat in CATEGORIES:
    run(f"""COPY INTO RAW.REVIEWS FROM @RAW.REVIEWS_STAGE/{cat}/
            FILE_FORMAT = (TYPE = PARQUET)
            MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
            ON_ERROR = ABORT_STATEMENT""")
    run(f"UPDATE RAW.REVIEWS SET category = '{cat}' WHERE category IS NULL")
    print(f"Loaded {cat}")

## 4. Check: rows in Snowflake must equal rows in the Parquet files

In [ ]:
expected = {cat: sum(pq.ParquetFile(f).metadata.num_rows
                     for f in (PARQUET_DIR / f"category={cat}").glob("*.parquet"))
            for cat in CATEGORIES}
loaded = dict(run("SELECT category, COUNT(*) FROM RAW.REVIEWS GROUP BY category").fetchall())

print(f"{'Category':<15}{'Parquet rows':>15}{'Snowflake rows':>17}")
for cat in CATEGORIES:
    print(f"{cat:<15}{expected[cat]:>15,}{loaded.get(cat, 0):>17,}")
assert all(loaded.get(c) == expected[c] for c in CATEGORIES), "Row counts do not match!"
print("\nRow counts match.")

## 5. Run dbt
`dbt run` builds the staging view and the two mart tables. `dbt test` runs the data-quality tests.
dbt reads the Snowflake settings from the environment variables set above (`dbt/profiles.yml` contains no secrets).

If `dbt` is not found, make sure you installed it in the same Python environment as this notebook (Python 3.10-3.12 works best with dbt).

In [ ]:
def dbt(*args):
    r = subprocess.run(["dbt", *args, "--project-dir", "dbt", "--profiles-dir", "dbt"],
                       capture_output=True, text=True)
    print(r.stdout[-5000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
    return r.returncode

if dbt("run") != 0:
    raise RuntimeError("dbt run failed, see the output above")

In [ ]:
dbt("test")   # a failing test is printed above; review it before continuing

In [ ]:
dbt("docs", "generate")   # optional: builds the documentation site; view it with: dbt docs serve --project-dir dbt --profiles-dir dbt

## 6. Look at the dbt marts

In [ ]:
def q(sql):
    cur.execute(sql)
    return cur.fetch_pandas_all()

print("Rows per category and split in mart_model_data:")
print(q("""SELECT category, split_in_category, split_temporal, COUNT(*) AS n
           FROM ANALYTICS.MART_MODEL_DATA GROUP BY 1, 2, 3 ORDER BY 1, 2, 3""").to_string(index=False))

print("\nmart_rating_stats (first rows):")
print(q("SELECT * FROM ANALYTICS.MART_RATING_STATS ORDER BY category, review_year LIMIT 8").round(3).to_string(index=False))

## 7. Export the modeling table for Stage 4
Stage 4 detects `data/processed/model_data.parquet` and uses it (with the dbt train/test flags) instead of sampling in Spark.

In [ ]:
model_data = q("SELECT * FROM ANALYTICS.MART_MODEL_DATA")
model_data.columns = [c.lower() for c in model_data.columns]
model_data.to_parquet("data/processed/model_data.parquet", index=False)
print(f"Saved data/processed/model_data.parquet: {len(model_data):,} rows")
print(model_data.groupby("category").size())

cur.close(); conn.close()

**Done with Stage 3.** Next: run `04_eda_modeling.ipynb`.

**Cost note:** the warehouse is XSMALL and suspends itself after 60 seconds idle, so a trial account's credits last easily. You can also run `ALTER WAREHOUSE REVIEWS_WH SUSPEND;` when you finish.

**For the report:** describe the flow as *Spark -> Parquet -> Snowflake RAW -> dbt staging -> dbt marts (+ tests and docs) -> modeling*. A screenshot of the `dbt test` output and the dbt docs lineage graph make good evidence.